# 📗 핑퐁·재시도·타임아웃

교안 01에서는 trace를 읽어 틀린 답의 원인을 찾았습니다. 이 교안은 팀이 **끝나지 않거나, 결과 없이 멈추거나, 외부 API가 잠깐 끊기거나 응답하지 않는** 경우를 다룹니다. 예시는 생활가전 브랜드 **온결**(수업용 가상 회사) 마케팅팀의 **주간 시장 브리핑 팀**입니다. 매주 공기청정기 AP-500의 경쟁사 가격, 업계 뉴스, 고객 리뷰를 모아 브리핑을 씁니다. 이 팀이 쓰는 외부 API는 일부러 실패하게 만들어 두었습니다.

<img src="images/00_lesson02_goals.png" width="1050" alt="오늘 배울 것 세 가지: 1 핑퐁·데드락 멈추기(작성자와 검토자가 같은 서류를 끝없이 주고받자 팀장이 계수기를 보고 멈춤), 2 일시 오류는 다시 시도(빠진 플러그를 다시 꽂아 성공, 망가진 서류처럼 다시 해도 안 되는 오류는 다시 시도하지 않음), 3 응답 없으면 시간 상한으로 끊기(초시계 시간이 지나면 기다림을 끊고 다음 일로)">

그림의 번호 1~3이 아래 학습 목표 1~3입니다.

**학습 목표**

1. **핑퐁과 에러 없는 중단을 찾아 막습니다.** `recursion_limit`, 라운드 가드, `RemainingSteps`, `defer=True`를 씁니다.
2. **일시 오류는 `RetryPolicy`로 다시 시도하고, 재시도하면 안 되는 오류를 가립니다.** `retry_on`에 일시 오류만 적습니다.
3. **응답 없는 호출은 노드 `timeout`으로 끊고 재시도 예산을 정합니다.** OpenAI SDK와 노드의 재시도가 겹쳐 시도 수가 곱해지지 않게 합니다.

## 1. 멀티에이전트는 어떻게 실패할까

교안 01의 MAST 분류는 역할·흐름 설계에서 생기는 실패를 다뤘습니다. 이 교안은 설계가 맞아도 **운영 중에** 생기는 실패를 다룹니다.

**운영 실패**는 그래프가 예외로 끝나거나, 끝나지 않거나, 끝났지만 결과 일부가 비어 있거나, 외부 시스템에 같은 일을 두 번 하는 경우입니다.

<img src="images/04_failure_map.png" width="1050" alt="멀티에이전트 실패 다섯 가지: 작성과 검토 담당이 반려된 초안을 계속 주고받는 무한 반복·핑퐁, 뉴스 API를 수화기를 든 채 기다리는 응답 없음, 가격 API 플러그가 잠깐 빠진 일시 오류, 조사원 세 명 중 리뷰 조사원만 넘어진 부분 실패, 게시판에 같은 주간 브리핑이 두 장 붙은 중복 부작용.">

| 실패 | 브리핑 팀 예 | 관측에서 보이는 신호 | 막는 장치 | 절 |
|---|---|---|---|---|
| 무한 반복·핑퐁 | 작성 담당과 검수 담당이 같은 반려 사유를 주고받음 | 같은 두 노드가 궤적에 번갈아 반복, 끝에 `GraphRecursionError` | `recursion_limit`, 라운드 가드, `RemainingSteps` | 3 |
| 응답 없음 | 뉴스 API가 응답하지 않음 | 노드 하나의 지연만 계속 늘어남 | async 노드 `timeout` + 재시도 | 5 |
| 일시 오류 | 가격 API 연결 끊김 | 같은 노드가 ERROR 뒤 성공 | `RetryPolicy`. 재시도로도 회복되지 않으면 `error_handler` 폴백 | 4, 교안 03의 3 |
| 부분 실패 | 리뷰 API 형식 변경으로 리뷰 조사만 실패 | 병렬 분기 하나가 ERROR, 다음 노드 기록 없음 | 조사 노드 안 `try/except` + 실패 기록 | 교안 03의 2 |
| 중복 부작용 | 다시 실행해 게시판에 브리핑 두 번 | 게시 호출과 게시판 글이 두 번 | 체크포인터로 멈춘 단계부터 다시 실행(이어 하기) + 멱등 키 | 교안 03의 4 |

## 2. 실습 준비

### 2-1. 라이브러리·모델·Langfuse 설정

`.env`에 `OPENAI_API_KEY`와 `LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY`, `LANGFUSE_BASE_URL`을 설정하고 새 커널에서 시작합니다. Langfuse 키 발급은 교안 01의 2-1, 화면 여는 법은 교안 01의 5-4와 같습니다.

라이브러리·모델·경로 설정과 실행 설정 함수 `make_config`는 교안 01과 같아서 한 셀에 모았습니다. `make_config`에만 `recursion_limit` 인자를 더했습니다. 3절에서 단계 상한을 바꿔 가며 실행하기 때문입니다. `trace_url`은 교안 01 5-3에서 trace 주소를 얻던 `flush()`와 `get_trace_url`을 함수 하나로 묶은 것입니다. 시연마다 `session_id`와 `thread_id`에 같은 값을 넘깁니다.

In [ ]:
# 실습에 필요한 라이브러리를 불러옵니다.
import os
import time
import asyncio
import operator
from datetime import datetime
from pathlib import Path
from typing import Annotated, Literal
from typing_extensions import TypedDict
from dotenv import find_dotenv, load_dotenv
from langchain_openai import ChatOpenAI
from langfuse import get_client
from langfuse.langchain import CallbackHandler
from IPython.display import Image

# material_dir는 output이 있는 단원 폴더입니다.
material_dir = Path(".")
data_dir = material_dir / "data"
output_dir = material_dir / "output"
output_dir.mkdir(parents=True, exist_ok=True)

# 단원 폴더에서 가까운 .env를 찾아 API 키를 읽습니다.
load_dotenv(find_dotenv(usecwd=True))
# 단원 .env에 없는 공통 키는 저장소 최상위 .env에서 읽습니다. 이미 읽은 키는 덮어쓰지 않습니다.
load_dotenv(material_dir.resolve().parent / ".env")

# 답변 모델: .env에 OPENAI_CHAT_MODEL이 없으면 기본 모델을 씁니다.
model_name = os.getenv("OPENAI_CHAT_MODEL", "gpt-5.6-luna")
llm = ChatOpenAI(model=model_name, use_responses_api=True)
print("답변 모델:", model_name)

# Langfuse 클라이언트: LANGFUSE_* 환경 변수로 연결합니다. 키가 틀리면 False가 나옵니다.
langfuse = get_client()
print("Langfuse 인증:", langfuse.auth_check())

# 노트북을 다시 실행해도 세션·thread_id 이름이 겹치지 않게 실행 시각을 붙입니다.
run_stamp = datetime.now().strftime("%m%d-%H%M%S")


# [보조 함수] 실행 설정 만들기: 실행마다 새 Langfuse 핸들러와 세션·thread_id·태그를 담습니다.
def make_config(session_id, thread_id, tags, recursion_limit=30):
    """Langfuse 콜백·세션·thread_id·태그·단계 상한을 담은 그래프 실행 설정을 만듭니다."""
    return {
        # 실행마다 새 핸들러를 넣어, 실행이 끝난 뒤 그 실행의 trace id를 핸들러에서 읽습니다.
        "callbacks": [CallbackHandler()],
        # trace 이름: Langfuse 목록에서 브리핑 팀 실행을 이 이름으로 찾습니다.
        "run_name": "briefing-team",
        # 그래프가 실행할 수 있는 단계 수의 상한입니다. config 최상위 키입니다.
        "recursion_limit": recursion_limit,
        # thread_id는 체크포인터가 State를 저장하는 열쇠입니다. 실패 뒤 멈춘 단계부터 다시 실행할 때 이 값으로 State를 찾습니다.
        "configurable": {"thread_id": thread_id},
        # langfuse_로 시작하는 키는 trace의 세션·태그로 들어갑니다.
        "metadata": {"langfuse_session_id": session_id, "langfuse_tags": tags},
    }


# [보조 함수] trace 주소: 보내지 않은 기록을 먼저 보내고 마지막 trace의 주소를 돌려줍니다.
def trace_url(config):
    """실행에 쓴 핸들러의 마지막 trace 주소를 반환합니다."""
    # Langfuse는 기록을 모아서 보내므로 flush 전에는 화면에 trace가 없을 수 있습니다.
    langfuse.flush()
    handler = config["callbacks"][0]
    return langfuse.get_trace_url(trace_id=handler.last_trace_id)

### 2-2. 브리핑 자료와 외부 API 함수

경쟁사 8곳의 공기청정기 가격, 업계 뉴스 3건, AP-500 고객 리뷰 3건을 노트북 안에 둡니다. 회사·제품·기사는 모두 수업용으로 만든 가상 자료입니다. 자료와 외부 API 함수는 오늘 배울 장치가 아니라 시연용 준비물이라 한 셀에 모았습니다.

**외부 API 함수:** 세 함수는 **수업용으로 일부러 실패하게 만든 함수**입니다. 실제 서비스라면 HTTP 요청이 들어갈 자리입니다.

| 함수 | 하는 일 | 넣어 둔 실패 (실패 스위치) |
|---|---|---|
| `fetch_prices()` | 경쟁사 가격 목록 | 앞의 `price_failures`번은 `ConnectionError("가격 API 연결 끊김")` (일시 오류) |
| `fetch_news()` (async) | 업계 뉴스 | 앞의 `news_hangs`번은 30초 동안 응답 없음 |
| `fetch_reviews()` | 고객 리뷰 | 스위치 없이 매번 `KeyError`. 예전 형식의 키 `"reviews"`를 읽기 때문입니다 (영구 오류) |

`CALLS`는 함수가 불린 횟수, `FAULTS`는 실패 스위치입니다. 스위치는 모두 꺼진 채로 시작합니다. **각 시연은 `reset_demo()`로 기록을 비운 뒤 그 시연에 필요한 실패 하나만 켜고 시작합니다.** 그래서 셀을 다시 실행해도 결과가 같습니다.

In [ ]:
# 이번 브리핑 주차(ISO 주차)와 우리 제품
WEEK = "2026-41"
OUR_PRODUCT = {"모델": "온결 AP-500", "가격": 429000}

# 경쟁사 8곳의 대표 공기청정기 가격. 금액 단위는 원입니다.
COMPETITOR_PRICES = [
    {"회사": "하늘가전", "모델": "클린맥스 S7", "이번 주": 389000, "지난주": 419000},
    {"회사": "바람전자", "모델": "에어핏 500", "이번 주": 349000, "지난주": 399000},
    {"회사": "맑음테크", "모델": "퓨어온 L", "이번 주": 449000, "지난주": 449000},
    {"회사": "숲결", "모델": "포레스트 360", "이번 주": 412000, "지난주": 412000},
    {"회사": "새벽홈", "모델": "던에어 7", "이번 주": 299000, "지난주": 319000},
    {"회사": "다온전자", "모델": "다온 클린 9", "이번 주": 469000, "지난주": 469000},
    {"회사": "이음가전", "모델": "이음 퓨어 S", "이번 주": 359000, "지난주": 379000},
    {"회사": "누리케어", "모델": "누리 에어 X", "이번 주": 398000, "지난주": 398000},
]

# 업계 뉴스 3건 (수업용 가상 기사)
NEWS = [
    {"날짜": "2026-10-05", "제목": "바람전자, 에어핏 500 가을 할인 10월 19일까지", "요약": "정가 399,000원에서 349,000원으로 50,000원 할인"},
    {"날짜": "2026-10-02", "제목": "하늘가전, 필터 정기 배송 구독 시작", "요약": "월 9,900원에 6개월마다 필터를 교체해 줍니다"},
    {"날짜": "2026-09-30", "제목": "10월 중순부터 미세먼지 나쁨 일수 늘어날 전망", "요약": "공기청정기 수요가 10월 셋째 주부터 늘 것으로 예상"},
]

# 리뷰 API의 새 응답 형식. 예전에는 {"reviews": [...]}였는데 {"data": {"items": [...]}}로 바뀌었습니다.
REVIEW_API_RESPONSE = {"data": {"items": [
    {"별점": 5, "내용": "소음이 작아 침실에 두고 씁니다."},
    {"별점": 4, "내용": "필터 교체 알림이 정확합니다."},
    {"별점": 2, "내용": "앱 연결이 자주 끊깁니다."},
]}}

# 함수별 호출 횟수와 실패 스위치(처음에는 모두 꺼짐)
CALLS = {"fetch_prices": 0, "fetch_news": 0, "fetch_reviews": 0}
FAULTS = {"price_failures": 0, "news_hangs": 0}


# [보조 함수] 시연 준비: 호출 수를 비우고 실패 스위치를 모두 끕니다.
def reset_demo():
    """CALLS를 0으로 되돌리고 FAULTS를 모두 끕니다."""
    for name in CALLS:
        CALLS[name] = 0

    # 앞 시연에서 켠 스위치가 다음 시연에 남지 않게 모두 끕니다.
    FAULTS["price_failures"] = 0
    FAULTS["news_hangs"] = 0


# [외부 API] 경쟁사 가격: 앞의 몇 번은 연결이 끊깁니다(일시 오류).
def fetch_prices():
    """경쟁사 가격 목록을 반환합니다. 앞의 price_failures번은 ConnectionError를 냅니다."""
    CALLS["fetch_prices"] += 1
    if CALLS["fetch_prices"] <= FAULTS["price_failures"]:
        raise ConnectionError("가격 API 연결 끊김")

    return COMPETITOR_PRICES


# [외부 API] 업계 뉴스: 앞의 몇 번은 응답하지 않습니다.
async def fetch_news():
    """업계 뉴스를 반환합니다. 앞의 news_hangs번은 30초 동안 아무것도 돌려주지 않습니다."""
    CALLS["fetch_news"] += 1
    if CALLS["fetch_news"] <= FAULTS["news_hangs"]:
        # time.sleep이 아니라 asyncio.sleep이어야 노드 timeout이 이 대기를 끊을 수 있습니다.
        await asyncio.sleep(30)

    return NEWS


# [외부 API] 고객 리뷰: 응답 형식이 바뀌어 매번 실패합니다(영구 오류).
def fetch_reviews():
    """고객 리뷰를 반환합니다. 예전 형식의 키를 읽어서 매번 KeyError를 냅니다."""
    CALLS["fetch_reviews"] += 1
    response = REVIEW_API_RESPONSE

    # 새 형식에는 "reviews" 키가 없습니다. 몇 번을 다시 불러도 같은 KeyError입니다.
    return response["reviews"]

## 3. 핑퐁과 데드락

**데드락**은 작업들이 서로의 결과를 기다리느라 아무도 진행하지 못하는 상태입니다. LangGraph 멀티에이전트 팀에서 노드끼리 서로를 기다리는 진짜 데드락은 드뭅니다. 실제로 보게 되는 것은 아래 두 모양이고, 데드락처럼 **결과가 나오지 않는다**는 점이 같아 함께 다룹니다.

| 모양 | 데드락과 비교 | 브리핑 팀 예 | 실행이 끝나는 모양 | 절 |
|---|---|---|---|---|
| 끝없이 주고받기 (핑퐁) | 기다리지 않고 계속 실행되지만 결과가 없음 | writer와 reviewer가 같은 반려 사유를 주고받음 | 계속 돌다가 `recursion_limit`에서 `GraphRecursionError` | 3-1, 3-2 |
| 기다리던 갈래가 오지 않음 (대기 엣지) | 노드 하나가 오지 않는 입력을 기다림 | 뉴스 수집을 건너뛰어 `assemble`이 실행되지 않음 | 에러 없이 끝나고 결과 필드가 비어 있음 | 3-3 |

### 3-1. 핑퐁 만들기

**핑퐁**은 두 노드가 서로에게 작업을 넘기기만 하고 끝나지 않는 반복입니다. 가장 흔한 원인은 **동시에 만족할 수 없는 조건**입니다.

<img src="images/05_ping_pong.png" width="1050" alt="writer가 8곳을 넣은 341자 초안과 3곳만 넣은 155자 초안을 번갈아 보내고, reviewer는 300자를 넘습니다와 빠진 경쟁사가 있습니다를 번갈아 돌려보내는 고리. 위 전광판에는 recursion_limit 10과 GraphRecursionError가 켜져 있고, 오른쪽에서 라운드 가드가 같은 사유 반복을 보고 고리를 멈춘 뒤 초안을 사람 확인 담당에게 넘깁니다.">

브리핑의 경쟁 가격 단락을 검수하는 규칙이 "300자 이내"와 "경쟁사 8곳 가격 모두 포함"인 경우:

1. writer가 8곳을 모두 넣어 씁니다. reviewer가 "300자를 넘습니다"로 반려합니다.
2. writer가 줄이려고 앞의 3곳만 남깁니다. reviewer가 "빠진 경쟁사가 있습니다"로 반려합니다.
3. writer가 다시 8곳을 모두 넣습니다. 1번으로 돌아갑니다.

LLM 없이 코드로 이 상황을 만듭니다. 두 노드 모두 `Command(goto)`(55일차)로 다음 노드를 정합니다. 검수 규칙은 3절의 세 검수 노드가 함께 쓰도록 `review_reason` 함수 하나로 둡니다. State·노드·연결은 55일차와 같은 방식이라 한 셀에서 그래프까지 만듭니다.

<img src="images/node_flow/l2_pingpong_overview.png" width="1050" alt="writer가 초안을 reviewer에게 보내고, reviewer는 반려하면 writer로, 통과하면 END로 보냅니다.">

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from langgraph.errors import GraphRecursionError


# [State] 가격 단락 작성 그래프
class DraftState(TypedDict):
    draft: str  # writer: 가격 단락
    feedback: str  # reviewer: 반려 사유


# [보조 함수] 가격 단락 만들기: 경쟁사 목록을 한 줄에 한 곳씩 씁니다.
def price_paragraph(prices):
    """경쟁사 가격 목록으로 브리핑의 가격 단락을 만듭니다."""
    lines = [f"[{WEEK}] 경쟁사 공기청정기 가격 (우리 AP-500 {OUR_PRODUCT['가격']:,}원)"]
    for item in prices:
        lines.append(f"{item['회사']} {item['모델']}: {item['이번 주']:,}원 (지난주 {item['지난주']:,}원)")

    return "\n".join(lines)


# [보조 함수] 검수 규칙: 두 규칙을 차례로 보고 반려 사유나 "통과"를 돌려줍니다. 두 규칙은 동시에 만족할 수 없습니다.
def review_reason(draft):
    """300자를 넘으면 길이 사유, 빠진 경쟁사가 있으면 누락 사유, 둘 다 아니면 "통과"를 반환합니다."""
    if len(draft) > 300:
        return "300자를 넘습니다"

    # 단락에 이름이 없는 경쟁사를 모읍니다.
    missing = []
    for item in COMPETITOR_PRICES:
        if item["회사"] not in draft:
            missing.append(item["회사"])

    if missing:
        return "빠진 경쟁사가 있습니다"
    return "통과"


# [노드] 가격 단락 작성: LLM 없이 반려 사유만 보고 고칩니다.
def draft_writer(state: DraftState) -> Command[Literal["reviewer"]]:
    """반려 사유가 길이 초과면 3곳만, 그 밖에는 8곳 모두로 단락을 씁니다."""
    # 길이로 반려되면 3곳만, 누락으로 반려되거나 첫 작성이면 8곳 모두 넣습니다. 그래서 두 단락이 번갈아 나옵니다.
    if state["feedback"] == "300자를 넘습니다":
        draft = price_paragraph(COMPETITOR_PRICES[:3])
    else:
        draft = price_paragraph(COMPETITOR_PRICES)

    return Command(goto="reviewer", update={"draft": draft})


# [노드] 가격 단락 검수: 통과면 끝내고, 아니면 사유를 적어 writer로 돌려보냅니다.
def draft_reviewer(state: DraftState) -> Command[Literal["writer", "__end__"]]:
    """review_reason의 판정대로 END나 writer로 보냅니다."""
    reason = review_reason(state["draft"])
    if reason == "통과":
        return Command(goto=END, update={"feedback": reason})

    return Command(goto="writer", update={"feedback": reason})


# 그래프 연결: writer와 reviewer가 Command로 서로를 부릅니다. 엣지는 START -> writer 하나뿐입니다.
builder = StateGraph(DraftState)
builder.add_node("writer", draft_writer)
builder.add_node("reviewer", draft_reviewer)
builder.add_edge(START, "writer")

pingpong_app = builder.compile()

# 컴파일한 노드와 연결을 이미지로 확인합니다.
display(Image(pingpong_app.get_graph().draw_mermaid_png()))

**실행 확인:** 두 길이를 먼저 봅니다. 8곳 모두 넣은 단락은 300자를 넘고, 3곳만 넣은 단락은 300자 안에 들어갑니다.

In [ ]:
# 두 규칙을 동시에 만족하는 단락이 없다는 것을 길이와 판정으로 확인합니다.
full_draft = price_paragraph(COMPETITOR_PRICES)
short_draft = price_paragraph(COMPETITOR_PRICES[:3])
print("8곳 단락 글자 수:", len(full_draft), "| 판정:", review_reason(full_draft))
print("3곳 단락 글자 수:", len(short_draft), "| 판정:", review_reason(short_draft))

**단계 상한**인 `recursion_limit`(50일차)을 넘으면 `GraphRecursionError`가 납니다. 정하지 않으면 기본값이 매우 커서 핑퐁을 사실상 막지 못합니다. LLM 노드끼리 핑퐁하면 상한보다 비용이 먼저 쌓이므로 **직접 정합니다.**

노드가 끝날 때마다 이름을 모으는 `stream_trail`로 상한 10에서 실행합니다. 예외가 나도 그때까지의 방문 순서가 남도록 `trail`은 밖에서 만들어 넘깁니다.

In [ ]:
# [보조 함수] 궤적 모으기: 노드가 끝날 때마다 이름을 trail에 넣고 마지막 State를 반환합니다.
def stream_trail(app, inputs, config, trail):
    """그래프를 실행하며 끝난 노드 이름을 trail에 순서대로 넣고 마지막 State를 반환합니다."""
    final = None
    for mode, chunk in app.stream(inputs, config, stream_mode=["updates", "values"]):
        if mode == "updates":
            # updates는 {노드 이름: 그 노드가 바꾼 값} 모양입니다.
            for node_name in chunk:
                trail.append(node_name)
        else:
            # values는 단계마다 State 전체입니다. 마지막 것이 최종 결과입니다.
            final = chunk

    return final

In [ ]:
# 시연마다 세션 하나: session_id와 thread_id에 같은 값을 넣고, 단계 상한은 10으로 낮춥니다.
pingpong_session = f"l2-pingpong-{run_stamp}"
pingpong_config = make_config(pingpong_session, pingpong_session, ["day56", "lesson02", "ping-pong"], recursion_limit=10)

# trail을 밖에서 만들어 넘기므로 예외가 나도 그때까지의 순서가 남습니다.
trail = []
try:
    stream_trail(pingpong_app, {"draft": "", "feedback": ""}, pingpong_config, trail)
except GraphRecursionError as error:
    print("예외:", type(error).__name__)
    # 메시지 뒤쪽은 문서 링크라 첫 줄만 출력합니다.
    print("메시지 첫 줄:", str(error).splitlines()[0])

print("방문 수:", len(trail), "| 방문 순서:", " -> ".join(trail))

**확인할 점:** `예외:`가 `GraphRecursionError`이고, `방문 순서:`는 `writer -> reviewer`가 5번 반복되어 `방문 수:`가 상한과 같은 10입니다. 예외 메시지는 상한에 닿았다는 것만 알려 줍니다. 어느 두 노드가 주고받았는지는 방문 순서로 확인합니다.

### 3-2. 상태로 막기: 라운드 가드와 같은 사유 감지

`recursion_limit`은 마지막 상한이라 닿는 순간 예외로 끝나고 작업 결과도 돌려받지 못합니다. 55일차의 **라운드 가드**처럼 State에 횟수를 두면 상한 전에 정상 경로로 끝낼 수 있습니다. 여기에 **같은 반려 사유가 다시 나오면** 바로 사람에게 넘기는 규칙을 더합니다. 같은 사유가 다시 나왔다는 것은 고친 단락이 앞에서 반려된 단락으로 돌아갔다는 뜻이기 때문입니다.

1. 반려 사유 "300자를 넘습니다"를 `reasons`에 기록하고 writer로 보냅니다.
2. 반려 사유 "빠진 경쟁사가 있습니다"를 기록하고 writer로 보냅니다.
3. "300자를 넘습니다"가 다시 나옵니다. `reasons`에 이미 있으므로 `escalate`(사람 확인 요청) 노드로 보냅니다.

**escalate:** 업무에서 지금 단계에서 해결하지 못한 문제를 더 높은 권한이나 더 전문적인 사람에게 넘기는 것을 말합니다. 여기서는 에이전트끼리 주고받기를 멈추고 사람 확인으로 넘기는 노드입니다.

<img src="images/node_flow/l2_guard_overview.png" width="1050" alt="writer와 reviewer가 주고받다가 reviewer가 같은 반려 사유를 다시 보거나 라운드 상한에 닿으면 escalate로 보내고 END로 끝납니다.">

In [ ]:
# 라운드 상한: 같은 사유 반복이 없어도 이 횟수만큼 반려하면 사람에게 넘깁니다.
MAX_ROUNDS = 4


# [State] 라운드 가드를 붙인 가격 단락 작성 그래프
class GuardState(TypedDict):
    draft: str
    feedback: str
    rounds: int  # reviewer: 지금까지 반려한 횟수
    reasons: Annotated[list[str], operator.add]  # reviewer: 지금까지의 반려 사유
    status: str  # reviewer·escalate: 끝난 이유


# [노드] 라운드 가드를 붙인 검수: 반려할 때마다 사유를 기록하고, 같은 사유가 다시 나오면 멈춥니다.
def guarded_reviewer(state: GuardState) -> Command[Literal["writer", "escalate", "__end__"]]:
    """review_reason으로 판정하고, 반려 사유 반복이나 라운드 상한이면 escalate로 보냅니다."""
    reason = review_reason(state["draft"])
    if reason == "통과":
        return Command(goto=END, update={"status": "통과"})

    # 같은 사유가 다시 나오면 앞에서 반려된 단락으로 돌아간 것입니다. 사람에게 넘깁니다.
    if reason in state["reasons"]:
        return Command(goto="escalate", update={"status": f"같은 반려 사유 반복: {reason}"})

    # 사유가 매번 달라도 반려가 상한만큼 쌓이면 사람에게 넘깁니다.
    if state["rounds"] >= MAX_ROUNDS:
        return Command(goto="escalate", update={"status": f"라운드 상한 {MAX_ROUNDS} 도달"})

    # 리듀서 필드 reasons에는 새 사유만 반환합니다.
    return Command(goto="writer", update={"feedback": reason, "rounds": state["rounds"] + 1, "reasons": [reason]})


# [노드] 사람 확인 요청: 마지막 단락과 끝난 이유를 남기고 끝냅니다.
def escalate(state: GuardState):
    """사람이 검수 규칙을 정리하도록 상태 문장 앞에 표시를 붙입니다."""
    return {"status": "사람 확인 필요: " + state["status"]}

In [ ]:
# writer는 3-1의 draft_writer를 그대로 쓰고, reviewer만 바꿉니다.
builder = StateGraph(GuardState)
builder.add_node("writer", draft_writer)
builder.add_node("reviewer", guarded_reviewer)
builder.add_node("escalate", escalate)
builder.add_edge(START, "writer")
builder.add_edge("escalate", END)

guard_app = builder.compile()

# 컴파일한 노드와 연결을 이미지로 확인합니다.
display(Image(guard_app.get_graph().draw_mermaid_png()))

In [ ]:
guard_session = f"l2-guard-{run_stamp}"
guard_config = make_config(guard_session, guard_session, ["day56", "lesson02", "round-guard"], recursion_limit=10)

# 3-1과 같은 상한 10으로 실행해도 예외 없이 끝납니다.
trail = []
guard_result = stream_trail(guard_app, {"draft": "", "feedback": "", "rounds": 0, "reasons": [], "status": ""}, guard_config, trail)

print("방문 수:", len(trail), "| 방문 순서:", " -> ".join(trail))
print("반려 사유 기록:", guard_result["reasons"])
print("처리 상태:", guard_result["status"])

**확인할 점:** `방문 순서:`가 writer와 reviewer를 세 번 오간 뒤 `escalate`에서 끝납니다(방문 7번). 3-1과 같은 상한 10에서 예외 없이 끝났고, `처리 상태:`는 "같은 반려 사유 반복: 300자를 넘습니다"입니다. 세 번째 반려 사유가 `반려 사유 기록`에 이미 있어서 reviewer가 `escalate`로 보냈습니다. 반려는 두 번뿐이라 라운드 상한(`MAX_ROUNDS` = 4)에는 닿지 않았습니다.

**남은 단계 수**를 담는 `RemainingSteps`는 State 필드의 타입으로 적으면 입력에 넣지 않아도 LangGraph가 값을 채우는 필드로, 매 단계 `recursion_limit`까지 남은 단계 수가 들어옵니다. 남은 단계가 2 이하이면 라운드 가드와 같은 `escalate`로 보내 예외 없이 끝낼 수 있습니다. 횟수를 직접 세지 않아도 되므로 반복 구조를 미리 알기 어려운 그래프에 씁니다. 3-1의 핑퐁 그래프에 붙여 봅니다.

In [ ]:
from langgraph.managed import RemainingSteps


# [State] 남은 단계 수를 받는 가격 단락 작성 그래프
class StepBudgetState(TypedDict):
    draft: str
    feedback: str
    status: str  # reviewer·escalate: 끝난 이유
    remaining_steps: RemainingSteps  # LangGraph가 매 단계 채웁니다. 입력에 넣지 않습니다.


# [노드] 남은 단계를 보는 검수: 한도 직전이면 판정하지 않고 escalate로 보냅니다.
def budget_reviewer(state: StepBudgetState) -> Command[Literal["writer", "escalate", "__end__"]]:
    """남은 단계가 2 이하이면 escalate로, 아니면 review_reason 판정대로 보냅니다."""
    # 2를 남기는 이유: 이 노드 다음에 escalate가 실행될 한 단계가 있어야 상한 전에 끝납니다.
    if state["remaining_steps"] <= 2:
        return Command(goto="escalate", update={"status": f"남은 단계 {state['remaining_steps']}에서 멈춤"})

    reason = review_reason(state["draft"])
    if reason == "통과":
        return Command(goto=END, update={"feedback": reason})
    return Command(goto="writer", update={"feedback": reason})


# writer는 3-1의 draft_writer, escalate는 라운드 가드의 escalate를 그대로 쓰고, reviewer만 바꿉니다.
builder = StateGraph(StepBudgetState)
builder.add_node("writer", draft_writer)
builder.add_node("reviewer", budget_reviewer)
builder.add_node("escalate", escalate)
builder.add_edge(START, "writer")
builder.add_edge("escalate", END)

budget_app = builder.compile()

# 컴파일한 노드와 연결을 이미지로 확인합니다.
display(Image(budget_app.get_graph().draw_mermaid_png()))

In [ ]:
budget_session = f"l2-steps-{run_stamp}"
budget_config = make_config(budget_session, budget_session, ["day56", "lesson02", "remaining-steps"], recursion_limit=10)

# 3-1과 같은 상한 10입니다. 이번에는 예외가 아니라 escalate로 끝납니다.
trail = []
budget_result = stream_trail(budget_app, {"draft": "", "feedback": "", "status": ""}, budget_config, trail)

print("방문 수:", len(trail), "| 마지막 노드:", trail[-1])
print("처리 상태:", budget_result["status"])

**확인할 점:** `마지막 노드:`가 `escalate`이고 `방문 수:`는 상한 10보다 적은 9입니다. 예외 없이 끝났고, `처리 상태:`는 "사람 확인 필요: 남은 단계 2에서 멈춤"입니다. 남은 단계가 2가 되자 reviewer가 판정 대신 `escalate`로 보냈습니다.

| 방법 | 언제 멈추나 | 끝나는 모양 | 맞는 경우 |
|---|---|---|---|
| `recursion_limit` | 단계 수가 상한을 넘을 때 | `GraphRecursionError` 예외 | 모든 실행에 마지막 상한으로 |
| 라운드 가드 + 같은 사유 감지 | 반려 횟수 상한, 같은 사유 반복 | `escalate`로 정상 종료, `status`에 이유 | 반복 구조(검수 루프)를 알 때 |
| `RemainingSteps` | 남은 단계가 정한 값 이하일 때 | `escalate`로 정상 종료, `status`에 이유 | 반복 구조를 미리 알기 어려울 때 |

### 3-3. 에러 없는 중단: 대기 엣지

**대기 엣지**는 `add_edge(["prices", "news_summary"], "assemble")`처럼 출발 노드를 목록으로 주는 엣지입니다. 목록의 노드가 **모두** 끝나야 `assemble`을 실행하므로, 여러 갈래를 기다렸다가 한 노드로 합칠 때 씁니다.

브리핑 팀의 수집 단계는 가격 갈래(`prices` 1단계)와 뉴스 갈래(`news_fetch` 다음 `news_summary`, 2단계)로 나뉘고, 두 갈래의 기록을 `assemble`이 묶어 `briefing`을 만듭니다. 뉴스 수집을 쉬는 주(`news_enabled=False`)에는 조건 분기가 뉴스 갈래 대신 `skip_news`로 갑니다. 같은 노드와 같은 두 입력으로 `assemble`을 합치는 방법만 바꿔 세 그래프를 비교합니다.

- **A `wait_app`:** `prices`와 `news_summary`를 대기 엣지로 `assemble`에 잇습니다.
- **B `normal_app`:** 갈래의 끝 노드 셋(`prices`, `news_summary`, `skip_news`)에서 `assemble`로 보통 엣지를 하나씩 잇습니다.
- **C `defer_app`:** B와 엣지가 같고, `assemble`만 `defer=True`로 등록합니다.

수집 노드는 기록 한 줄만 남기는 단순한 함수라 State·노드·라우팅 함수, 세 그래프가 같이 쓰는 노드·엣지, 두 입력을 실행하는 함수를 한 셀에 모았습니다.

<img src="images/node_flow/l2_silent_overview.png" width="1050" alt="START에서 prices로 가는 고정 엣지와, 조건 분기로 news_fetch 또는 skip_news로 가는 엣지가 있습니다. news_fetch 다음은 news_summary입니다. assemble은 prices와 news_summary가 모두 끝나야 실행되는 대기 엣지로 연결되어 있고, skip_news 뒤에는 엣지가 없습니다.">

In [ ]:
# [State] 수집 단계 그래프
class CollectState(TypedDict):
    news_enabled: bool  # 이번 주 뉴스 수집 여부
    log: Annotated[list[str], operator.add]  # 노드마다 한 줄씩 남기는 기록
    briefing: str  # assemble: 묶은 결과


# [노드] 수집 노드 넷과 묶는 노드 하나. 합치는 방법의 차이만 보려고 외부 API 대신 기록만 남깁니다.
def collect_prices_step(state: CollectState):
    """가격 수집을 마쳤다고 기록합니다."""
    return {"log": ["가격 수집"]}


def fetch_news_step(state: CollectState):
    """뉴스 수집을 마쳤다고 기록합니다."""
    return {"log": ["뉴스 수집"]}


def summarize_news_step(state: CollectState):
    """뉴스 요약을 마쳤다고 기록합니다."""
    return {"log": ["뉴스 요약"]}


def skip_news(state: CollectState):
    """이번 주 뉴스 수집을 건너뛰었다고 기록합니다."""
    return {"log": ["뉴스 생략"]}


def assemble(state: CollectState):
    """지금까지의 기록을 묶어 briefing을 만듭니다."""
    return {"briefing": " / ".join(state["log"])}


# [라우팅 함수] 뉴스 수집 여부로 갈래를 고릅니다.
def route_news(state: CollectState):
    """news_enabled면 news_fetch, 아니면 skip_news를 반환합니다."""
    if state["news_enabled"]:
        return "news_fetch"
    return "skip_news"


def add_collect_branches(builder):
    """세 그래프가 같이 쓰는 부분: 수집 노드 넷과 START에서 갈래 끝까지의 엣지를 넣습니다."""
    builder.add_node("prices", collect_prices_step)
    builder.add_node("news_fetch", fetch_news_step)
    builder.add_node("news_summary", summarize_news_step)
    builder.add_node("skip_news", skip_news)
    builder.add_edge(START, "prices")
    builder.add_conditional_edges(START, route_news, ["news_fetch", "skip_news"])
    builder.add_edge("news_fetch", "news_summary")


def run_both_weeks(app, tag):
    """뉴스를 수집하는 주와 쉬는 주를 차례로 실행하고, 방문 순서와 assemble이 만든 briefing을 출력합니다."""
    for news_enabled in [True, False]:
        # 실행마다 새 설정(새 핸들러)을 만들어 trace가 섞이지 않게 합니다.
        session = f"l2-{tag}-{run_stamp}"
        config = make_config(session, session, ["day56", "lesson02", tag])

        trail = []
        briefings = []  # assemble이 실행될 때마다 만든 briefing
        # updates는 {노드 이름: 그 노드가 바꾼 값} 모양이라 assemble이 실행될 때마다 그 결과를 모을 수 있습니다.
        for chunk in app.stream({"news_enabled": news_enabled, "log": []}, config, stream_mode="updates"):
            for node_name, update in chunk.items():
                trail.append(node_name)
                if node_name == "assemble":
                    briefings.append(update["briefing"])

        print("뉴스 수집:", news_enabled, "| 방문 순서:", " -> ".join(trail))
        print(f"    assemble 실행 {len(briefings)}번 | 만든 briefing: {briefings}")

**A. 대기 엣지로 합치기**

In [ ]:
builder = StateGraph(CollectState)
add_collect_branches(builder)
builder.add_node("assemble", assemble)

# 대기 엣지: prices와 news_summary가 모두 끝나야 assemble이 실행됩니다.
builder.add_edge(["prices", "news_summary"], "assemble")
builder.add_edge("assemble", END)

wait_app = builder.compile()

# 컴파일한 노드와 연결을 이미지로 확인합니다.
display(Image(wait_app.get_graph().draw_mermaid_png()))

In [ ]:
# 뉴스를 수집하는 주와 쉬는 주를 차례로 실행합니다.
run_both_weeks(wait_app, "wait-edge")

**확인할 점:** `뉴스 수집: True` 줄은 `assemble 실행 1번`이고 briefing에 세 기록이 모두 있습니다. `뉴스 수집: False` 줄은 방문 순서가 `prices -> skip_news`에서 끝나고 `assemble 실행 0번`, `만든 briefing: []`입니다. 셀은 예외 없이 끝났습니다. `news_summary`가 실행되지 않아 대기 엣지의 조건(`prices`와 `news_summary` 모두 끝남)이 채워지지 않았고, 더 실행할 노드가 없어 그래프가 정상 종료한 것입니다. 이런 중단은 궤적에 `assemble`이 있는지, `briefing`이 채워졌는지를 봐야 찾습니다.

**B. 보통 엣지로 합치기**

In [ ]:
builder = StateGraph(CollectState)
add_collect_branches(builder)
builder.add_node("assemble", assemble)

# 보통 엣지 셋: 어느 갈래로 가든 그 갈래의 끝 노드에서 assemble로 가는 엣지가 있습니다.
builder.add_edge("prices", "assemble")
builder.add_edge("news_summary", "assemble")
builder.add_edge("skip_news", "assemble")
builder.add_edge("assemble", END)

normal_app = builder.compile()

# 컴파일한 노드와 연결을 이미지로 확인합니다.
display(Image(normal_app.get_graph().draw_mermaid_png()))

In [ ]:
# A와 같은 두 입력으로 실행합니다.
run_both_weeks(normal_app, "normal-edge")

**확인할 점:** `뉴스 수집: False` 줄은 `assemble 실행 1번`, briefing `'가격 수집 / 뉴스 생략'`으로 A에서 빠졌던 쉬는 주의 결과가 만들어집니다. 대신 `뉴스 수집: True` 줄이 `assemble 실행 2번`이고, 첫 briefing `'뉴스 수집 / 가격 수집'`에는 `뉴스 요약`이 없습니다. 보통 엣지는 들어오는 엣지 하나만 신호를 보내도 다음 단계에 `assemble`을 실행합니다. 단계가 하나인 `prices`가 끝나자 `assemble`이 `news_summary`와 같은 단계에 실행되어 요약 전의 기록을 묶었고, `news_summary`가 끝난 뒤 한 번 더 실행되었습니다.

**`defer=True`로 미루기.** `add_node("assemble", assemble, defer=True)`처럼 등록하면 그 노드의 실행을 **그래프 실행이 끝나기 직전**까지 미룹니다.

- **작동:** 들어오는 엣지가 실행 신호를 보내도 바로 실행하지 않고 신호를 보관합니다. 실행할 다른 노드가 하나도 남지 않은 단계가 되면 보관한 신호로 그 노드를 실행합니다. 신호가 여러 번 와도 실행은 한 번입니다.
- **마지막 노드일 필요는 없습니다:** 미룬 노드가 실행된 뒤에는 그 노드에서 나가는 엣지를 따라 다음 노드가 평소대로 실행됩니다.
- **주의:** 합칠 갈래와 관계없는 노드까지 포함해, 이 실행에서 남은 작업이 모두 끝나야 실행됩니다. 노드를 계속 실행하는 반복이 있으면 반복이 끝날 때까지 기다립니다. `defer=True` 노드가 여럿이면 같은 단계에 함께 실행되므로 이들 사이의 순서는 `defer`로 정할 수 없습니다.
- **쓰는 곳:** 실행마다 갈래의 수나 길이가 달라지는 합류 지점입니다(조건 분기로 고른 갈래, `Send`로 나눈 작업, 단계 수가 다른 갈래).

**C. 보통 엣지 + `defer=True`**

In [ ]:
builder = StateGraph(CollectState)
add_collect_branches(builder)
# B와 다른 곳은 이 줄의 defer=True 하나입니다.
builder.add_node("assemble", assemble, defer=True)

builder.add_edge("prices", "assemble")
builder.add_edge("news_summary", "assemble")
builder.add_edge("skip_news", "assemble")
builder.add_edge("assemble", END)

defer_app = builder.compile()

# 컴파일한 노드와 연결을 이미지로 확인합니다.
display(Image(defer_app.get_graph().draw_mermaid_png()))

In [ ]:
# A·B와 같은 두 입력으로 실행합니다.
run_both_weeks(defer_app, "defer")

**확인할 점:** `뉴스 수집: True`와 `뉴스 수집: False` 두 줄 모두 `assemble 실행 1번`이고 방문 순서의 마지막이 `assemble`입니다. True 줄의 briefing은 `'뉴스 수집 / 가격 수집 / 뉴스 요약'`으로 두 갈래의 기록이 모두 있고, False 줄은 `'가격 수집 / 뉴스 생략'`입니다. B와 C는 `defer=True` 한 곳만 다르므로, B의 이른 첫 실행을 없앤 것은 `defer=True`입니다.

## 4. 일시 오류와 재시도

**일시 오류**는 같은 요청을 조금 뒤에 다시 보내면 성공하는 오류입니다(연결 끊김, 서버 과부하 응답 5xx). 반대로 **영구 오류**는 몇 번을 다시 보내도 같은 결과가 나오는 오류입니다(형식이 바뀐 응답, 잘못된 인자). 재시도는 일시 오류에만 겁니다.

가격 API가 앞의 두 번 연결이 끊기는 경우, 재시도를 걸면:

1. 가격 조사 노드가 `fetch_prices()`를 부릅니다. 연결 끊김(`ConnectionError`)으로 실패합니다.
2. 0.5초 뒤 다시 부릅니다. 또 실패합니다.
3. 1초 뒤 다시 부릅니다(대기 간격이 2배로 늘어남). 이번에는 성공합니다.

### 4-1. 재시도 없이 실행

재시도 없이 실행하면 첫 연결 끊김에서 그래프 전체가 `ConnectionError`로 끝나고 가격 API는 1번만 불립니다. 가격 조사 노드 하나만 있는 그래프로 확인합니다.

In [ ]:
from langgraph.types import RetryPolicy


# [State] 가격 조사 그래프
class PriceState(TypedDict):
    prices: list[dict]  # collect_prices: 경쟁사 가격 목록


# [노드] 가격 조사: 외부 API를 한 번 부릅니다. 재시도는 노드 함수가 아니라 add_node의 retry_policy가 맡습니다.
def collect_prices(state: PriceState):
    """경쟁사 가격 목록을 받아 prices에 넣습니다."""
    return {"prices": fetch_prices()}


builder = StateGraph(PriceState)
builder.add_node("collect_prices", collect_prices)
builder.add_edge(START, "collect_prices")
builder.add_edge("collect_prices", END)

no_retry_app = builder.compile()

# 컴파일한 노드와 연결을 이미지로 확인합니다.
display(Image(no_retry_app.get_graph().draw_mermaid_png()))

In [ ]:
reset_demo()
# 이 시연에서 켜는 실패: 가격 API 앞 2번 연결 끊김
FAULTS["price_failures"] = 2
retry_session = f"l2-retry-{run_stamp}"
retry_config = make_config(retry_session, retry_session, ["day56", "lesson02", "retry", "v1-no-retry"])

# 첫 호출이 실패하면 그래프 전체가 그 예외로 끝납니다.
try:
    no_retry_app.invoke({"prices": []}, retry_config)
except ConnectionError as error:
    print("예외:", type(error).__name__, error)

print("가격 API 호출 수:", CALLS["fetch_prices"])

### 4-2. `RetryPolicy` 붙이기

**재시도 규칙**인 `RetryPolicy`는 노드가 예외를 내면 같은 노드를 다시 실행하는 규칙입니다. `add_node(..., retry_policy=RetryPolicy(...))`로 노드마다 붙입니다. 노드 함수는 바꾸지 않습니다.

| 인자 | 기본값 | 뜻 |
|---|---|---|
| `max_attempts` | 3 | **첫 시도를 포함한** 최대 시도 횟수 |
| `initial_interval` | 0.5 | 첫 재시도 전 대기(초) |
| `backoff_factor` | 2.0 | 재시도마다 대기 간격에 곱하는 수 |
| `max_interval` | 128.0 | 대기 간격 상한(초) |
| `jitter` | True | 대기 간격에 0~1초 사이 무작위 값을 더함. 여러 실행이 같은 순간에 다시 몰리지 않게 합니다 |
| `retry_on` | 기본 판정 함수 | 재시도할 예외를 고르는 판정(아래 표) |

**`retry_on`에 넣을 수 있는 판정**

| 넣는 값 | 예 | 재시도하는 경우 |
|---|---|---|
| 예외 클래스 하나 | `retry_on=ConnectionError` | 그 예외나 하위 예외가 났을 때 |
| 예외 클래스 묶음(리스트나 튜플) | `retry_on=[ConnectionError, NodeTimeoutError]` 또는 `retry_on=(ConnectionError, NodeTimeoutError)` | 묶음 가운데 하나가 났을 때 |
| 판정 함수 | `retry_on=is_retryable` | 예외를 받아 `True`를 돌려줄 때(예: HTTP 상태 코드가 429나 5xx일 때만 `True`) |
| 넣지 않음 | 기본 판정 함수 | 연결 끊김, 서버 5xx 오류, `NodeTimeoutError`는 재시도하고 `ValueError`, `KeyError` 같은 코드·데이터 오류는 재시도하지 않음 |

`retry_on=ConnectionError`로 **연결 끊김만** 재시도하게 정하고, 4절 첫머리의 세 단계를 실행해 봅니다.

In [ ]:
# 같은 노드에 재시도 규칙만 붙입니다. 첫 시도 포함 3번까지, 연결 끊김만 다시 시도합니다.
builder = StateGraph(PriceState)
builder.add_node("collect_prices", collect_prices,
                 retry_policy=RetryPolicy(max_attempts=3, initial_interval=0.5, retry_on=ConnectionError))
builder.add_edge(START, "collect_prices")
builder.add_edge("collect_prices", END)

retry_app = builder.compile()

# 컴파일한 노드와 연결을 이미지로 확인합니다. 재시도는 그림에 나타나지 않습니다.
display(Image(retry_app.get_graph().draw_mermaid_png()))

In [ ]:
reset_demo()
# 이 시연에서 켜는 실패: 가격 API 앞 2번 연결 끊김(4-1과 같음)
FAULTS["price_failures"] = 2
retry_config = make_config(retry_session, retry_session, ["day56", "lesson02", "retry", "v2-retry"])

# 대기 간격 0.5초와 1초에 jitter(재시도마다 0~1초)가 더해져 1.5~3.5초쯤 걸립니다.
start = time.perf_counter()
retry_result = retry_app.invoke({"prices": []}, retry_config)
elapsed = time.perf_counter() - start

print("가격 API 호출 수:", CALLS["fetch_prices"], "| 받은 가격:", len(retry_result["prices"]), "건 | 걸린 시간:", round(elapsed, 1), "초")
print("trace:", trace_url(retry_config))

**Langfuse에서 확인**

Langfuse 화면 주소의 형식과 읽는 법은 교안 01의 5-4와 같습니다.

- 주소: `https://cloud.langfuse.com/project/<프로젝트 ID>/traces/<trace id>?observation=<항목 id>` (위에 출력된 trace 주소에서 첫 번째 collect_prices를 누른 주소)
- 누를 곳: 위에 출력된 trace 주소, 상세 화면의 **Tree** 탭, 첫 번째 **collect_prices**
- 보여 주는 것: 재시도한 노드는 시도마다 따로 기록됩니다. 빨간 상자 안의 앞 두 개에는 Error 표시가 붙고, 표시가 없는 마지막 하나가 성공한 시도입니다. Error 항목을 누르면 오른쪽 **Error** 칸에 예외 메시지(가격 API 연결 끊김)가 나옵니다.

<img src="images/langfuse/l2_retry_attempts.png" width="1050" alt="Langfuse trace 상세의 Tree 탭에서 collect_prices 세 개를 빨간 상자로 강조한 화면. 앞의 두 개에 Error 표시가 붙고 마지막 하나는 표시가 없으며, 첫 번째를 눌러 오른쪽 Error 칸에 가격 API 연결 끊김이 보인다">

**확인할 점:** `가격 API 호출 수:`가 3이고 `받은 가격:`이 8건입니다. 연결 끊김 2번 뒤 세 번째 시도에서 성공했습니다. 반환된 결과에는 실패한 두 시도가 남지 않으므로, 재시도가 있었는지는 trace의 Error 기록으로 확인합니다.

### 4-3. 재시도하면 안 되는 오류

`retry_on`을 정하지 않으면 LangGraph의 기본 판정을 씁니다. 기본 판정은 **코드나 데이터가 틀려서 나는 예외는 재시도하지 않습니다.**

| 예외 | 기본 판정 | 예 |
|---|---|---|
| `ConnectionError` | 재시도함 | 가격 API 연결 끊김 |
| HTTP 5xx 응답 오류 (`httpx`, `requests`) | 재시도함 | 서버 과부하 |
| `NodeTimeoutError` (노드 `timeout` 초과 시 나는 예외, 5절에서 사용) | 재시도함 | 뉴스 API 무응답 |
| `ValueError`, `TypeError`, `LookupError`(`KeyError`, `IndexError`) | 재시도 안 함 | 리뷰 API 형식 변경 |
| `RuntimeError`, `OSError` | 재시도 안 함 | |
| 위 목록에 없는 예외 (직접 만든 예외 등) | 재시도함 | |

시연용으로 `raise RuntimeError("일시 장애")`를 쓰면 기본 판정에서는 재시도되지 않습니다. 일시 오류를 재현할 때는 `ConnectionError`를 쓰거나 `retry_on`을 직접 정합니다.

리뷰 조사 노드에 `retry_on` 없이 `RetryPolicy(max_attempts=3)`을 붙여 봅니다.

In [ ]:
# [State] 리뷰 조사 그래프
class ReviewState(TypedDict):
    reviews: list[dict]  # collect_reviews: 고객 리뷰 목록


# [노드] 리뷰 조사: 외부 API를 한 번 부릅니다.
def collect_reviews(state: ReviewState):
    """고객 리뷰를 받아 reviews에 넣습니다."""
    return {"reviews": fetch_reviews()}


# retry_on을 정하지 않아 기본 판정을 씁니다.
builder = StateGraph(ReviewState)
builder.add_node("collect_reviews", collect_reviews, retry_policy=RetryPolicy(max_attempts=3))
builder.add_edge(START, "collect_reviews")
builder.add_edge("collect_reviews", END)

review_app = builder.compile()

# 컴파일한 노드와 연결을 이미지로 확인합니다.
display(Image(review_app.get_graph().draw_mermaid_png()))

In [ ]:
# 켜는 스위치 없음: 리뷰 API는 스위치 없이 늘 KeyError를 냅니다.
reset_demo()
review_session = f"l2-retry-{run_stamp}"
review_config = make_config(review_session, review_session, ["day56", "lesson02", "retry", "permanent-error"])

# KeyError는 기본 판정에서 재시도 대상이 아니므로 한 번 만에 끝납니다.
try:
    review_app.invoke({"reviews": []}, review_config)
except KeyError as error:
    print("예외:", type(error).__name__, error)

print("리뷰 API 호출 수:", CALLS["fetch_reviews"])

**확인할 점:** `예외:`가 `KeyError 'reviews'`이고, `max_attempts=3`인데 `리뷰 API 호출 수:`는 1입니다. 기본 판정이 `KeyError`를 재시도하지 않아 첫 실패에서 끝났습니다.

## 5. 타임아웃과 재시도 예산

**노드 타임아웃**은 노드 한 번의 시도가 쓸 수 있는 시간의 상한입니다. `add_node(..., timeout=초)`로 정하고, 넘으면 LangGraph가 그 시도를 취소하고 `NodeTimeoutError`를 냅니다. 응답하지 않는 API는 예외도 내지 않으므로 타임아웃이 없으면 팀 전체가 계속 기다립니다.

**재시도 예산**은 타임아웃과 재시도를 함께 걸었을 때 최악의 총 대기 시간입니다. 사용자가 기다릴 수 있는 시간 안에 들어오게 정합니다.

<img src="images/06_timeout_budget.png" width="1050" alt="위 줄은 타임아웃이 없어 응답 없는 뉴스 API를 30초 동안 기다리는 회색 막대이고, 아래 줄은 timeout=2, 시도 2번 설정으로 1번째 시도가 2초에서 NodeTimeoutError로 잘린 뒤 대기 0.5초를 거쳐 2번째 시도가 성공합니다. 맨 아래 공식 카드에는 최악 대기 = 시도 수 x 타임아웃 + 대기 합이 적혀 있습니다.">

뉴스 API가 첫 호출에 응답하지 않는 경우:

1. 뉴스 조사 노드의 첫 시도가 응답 없는 뉴스 API를 기다립니다.
2. 2초가 지나면 LangGraph가 그 시도를 끊고 `NodeTimeoutError`를 냅니다.
3. 0.5초 기다린 뒤 두 번째 시도를 합니다. 이번에는 뉴스 API가 바로 응답해 성공합니다.

**타임아웃은 async 노드에만 걸 수 있습니다.** 실행 중인 일반(sync) 함수는 파이썬 안에서 안전하게 중단할 수 없기 때문입니다. 일반 함수 노드에 `timeout`을 주면 `add_node`는 통과하지만 `compile()`에서 `ValueError`가 나서 실행 전에 막힙니다. 그래서 `fetch_news`와 이를 부르는 노드를 `async def`로 만들었습니다. async 노드 안에서 `time.sleep()`처럼 이벤트 루프를 막는 코드를 쓰면 타임아웃이 제때 동작하지 않으므로 `await asyncio.sleep()`을 씁니다.

### 5-1. 타임아웃만 걸기

async 노드가 있는 그래프는 `ainvoke`로 실행하고 노트북 셀에서 바로 `await`합니다.

In [ ]:
from langgraph.errors import NodeTimeoutError


# [State] 뉴스 조사 그래프
class NewsState(TypedDict):
    news: list[dict]  # collect_news: 업계 뉴스 목록


# [노드] 뉴스 조사: async 함수라야 timeout을 걸 수 있습니다.
async def collect_news(state: NewsState):
    """업계 뉴스를 받아 news에 넣습니다."""
    return {"news": await fetch_news()}


builder = StateGraph(NewsState)
# 시도 한 번에 2초까지만 기다립니다.
builder.add_node("collect_news", collect_news, timeout=2)
builder.add_edge(START, "collect_news")
builder.add_edge("collect_news", END)

timeout_app = builder.compile()

# 컴파일한 노드와 연결을 이미지로 확인합니다.
display(Image(timeout_app.get_graph().draw_mermaid_png()))

In [ ]:
reset_demo()
# 이 시연에서 켜는 실패: 뉴스 API 첫 호출 30초 무응답
FAULTS["news_hangs"] = 1
timeout_session = f"l2-timeout-{run_stamp}"
timeout_config = make_config(timeout_session, timeout_session, ["day56", "lesson02", "timeout", "v1-timeout-only"])

# 30초를 기다리지 않고 2초 만에 NodeTimeoutError로 끝납니다.
start = time.perf_counter()
try:
    await timeout_app.ainvoke({"news": []}, timeout_config)
except NodeTimeoutError as error:
    print("예외:", type(error).__name__, "| 노드:", error.node, "| 상한(초):", error.run_timeout)
elapsed = time.perf_counter() - start

print("걸린 시간:", round(elapsed, 1), "초 | 뉴스 API 호출 수:", CALLS["fetch_news"])

**확인할 점:** `예외:`가 `NodeTimeoutError`, `노드:`가 `collect_news`, `상한(초):`가 `add_node`에 준 2.0입니다. `걸린 시간:`은 30초가 아니라 2.0초이고 `뉴스 API 호출 수:`는 1입니다. 타임아웃이 응답 없는 첫 호출을 2초에서 끊었습니다.

### 5-2. 타임아웃과 재시도 함께 걸기

같은 노드에 `RetryPolicy`를 더합니다. 어떤 예외를 재시도하는지 코드에서 바로 보이도록 `retry_on=NodeTimeoutError`로 적습니다.

In [ ]:
builder = StateGraph(NewsState)
# 시도마다 2초 상한, 시간 초과면 0.5초 뒤 한 번 더 시도합니다(첫 시도 포함 2번).
builder.add_node("collect_news", collect_news, timeout=2,
                 retry_policy=RetryPolicy(max_attempts=2, initial_interval=0.5, retry_on=NodeTimeoutError))
builder.add_edge(START, "collect_news")
builder.add_edge("collect_news", END)

timeout_retry_app = builder.compile()

# 컴파일한 노드와 연결을 이미지로 확인합니다.
display(Image(timeout_retry_app.get_graph().draw_mermaid_png()))

In [ ]:
reset_demo()
# 이 시연에서 켜는 실패: 뉴스 API 첫 호출 30초 무응답(5-1과 같음)
FAULTS["news_hangs"] = 1
timeout_config = make_config(timeout_session, timeout_session, ["day56", "lesson02", "timeout", "v2-timeout-retry"])

# 첫 시도 2초 + 대기 0.5초 + jitter(0~1초) + 두 번째 시도(바로 응답)이므로 2.5~3.5초쯤 걸립니다.
start = time.perf_counter()
news_result = await timeout_retry_app.ainvoke({"news": []}, timeout_config)
elapsed = time.perf_counter() - start

print("뉴스 API 호출 수:", CALLS["fetch_news"], "| 받은 뉴스:", len(news_result["news"]), "건 | 걸린 시간:", round(elapsed, 1), "초")

**확인할 점:** `뉴스 API 호출 수:`가 2이고 `받은 뉴스:`가 3건입니다. 첫 시도가 2초에서 `NodeTimeoutError`로 끊긴 뒤 두 번째 시도가 성공했습니다. `걸린 시간:`은 jitter 때문에 실행마다 다르고 2.5~3.5초 사이입니다.

재시도 예산은 모든 시도가 타임아웃까지 간 최악의 경우로 셉니다. 그림 공식에 기본으로 켜진 `jitter`(재시도마다 최대 1초)를 더합니다.

**최악 대기 = 시도 수 x 타임아웃 + 시도 사이 대기 간격의 합 + jitter 최대값(재시도마다 1초)**

위 설정이면 2 x 2초 + 0.5초 + 1초 = 5.5초입니다. 이번 실행이 그보다 짧게 끝난 것은 두 번째 시도가 바로 성공했기 때문입니다.

| 정할 것 | 기준 | 브리핑 팀 예 |
|---|---|---|
| 타임아웃 | 정상 응답 시간보다 넉넉하게. 너무 짧으면 정상 호출도 끊음 | 뉴스 API가 평소 1초 안에 응답하면 2~5초 |
| 시도 수 | 일시 오류가 풀리는 데 드는 횟수 | 2~3번 |
| 총 예산 | 사용자나 다음 단계가 기다릴 수 있는 시간 | 주간 브리핑은 몇 분도 괜찮음, 채팅 답변은 수십 초 안 |
| 예산을 넘으면 | 재시도를 멈추고 실패로 기록하거나 폴백 | 뉴스 없이 브리핑을 쓰고 빠진 자료를 밝힘 |

LLM을 부르는 노드에 타임아웃을 걸 때는 모델 응답 시간까지 넣어 정합니다. 위의 2초는 외부 API 하나만 부르는 노드라서 쓸 수 있는 값입니다.

### 5-3. 겹친 재시도: 계층마다 곱해지는 시도 수

재시도는 LangGraph 노드에만 있는 것이 아닙니다. 이 교안의 `llm`(`ChatOpenAI`)이 요청을 보낼 때 쓰는 OpenAI 파이썬 SDK도 실패한 요청을 스스로 다시 보냅니다. `max_retries`를 정하지 않으면 SDK 기본값 2(재시도 2번, 곧 첫 시도 포함 시도 3번)를 써서 **모델 호출 한 번이 최대 3번의 HTTP 요청**이 됩니다. SDK가 다시 보내는 오류는 연결 오류, 408(요청 시간 초과), 409(충돌), 429(요청 한도 초과), 5xx(서버 오류)입니다(OpenAI Python SDK README의 Retries).

**겹친 재시도**는 여러 계층이 같은 실패를 각자 재시도해 시도 수가 곱해지는 것입니다. 노드 `RetryPolicy`는 노드 전체를 다시 실행하므로, 다시 실행된 노드 안의 모델 호출은 SDK 재시도를 처음부터 다시 합니다. SDK 시도 3번에 노드 `RetryPolicy(max_attempts=2)`의 시도 2번이 겹치면 모델 서버에는 최대 3 x 2 = 6번 요청이 갑니다.

<img src="images/10_nested_retry.png" width="1050" alt="RetryPolicy(max_attempts=2) 상자 안의 노드 시도 1과 노드 시도 2가 각각 OpenAI SDK 시도 3번을 다시 시작해 모델 서버에 3 x 2 = 최대 6번 요청이 갑니다. 오른쪽 Langfuse 화면에는 노드 시도마다 하나씩 generation 2개만 보이고, 재시도는 지연만 길어진 것으로 나타납니다.">

| 계층 | 누가 다시 시도하나 | 다시 하는 범위 | Langfuse에 보이는 모양 |
|---|---|---|---|
| OpenAI SDK 내부 재시도 | OpenAI 클라이언트 (`max_retries` 기본 2 = 시도 3번) | HTTP 요청 하나 | generation 하나. 재시도 횟수는 보이지 않고 지연만 길어짐 |
| 노드 `RetryPolicy` | LangGraph (`max_attempts`, 첫 시도 포함) | 노드 전체 (외부 API 호출 + 모델 호출) | 같은 노드 기록이 시도마다 따로 남음 (4-2) |

SDK 재시도는 Langfuse에 따로 남지 않아, trace의 generation 수만 세면 실제 요청 수를 적게 셉니다.

**노드도 다시 시도할지는 노드의 `retry_on`이 정합니다.** SDK가 3번 시도하고도 실패하면 `openai.APIConnectionError`, `openai.RateLimitError` 같은 예외를 노드 밖으로 올립니다. 노드의 `RetryPolicy`가 이 예외를 재시도 대상으로 보면 노드를 다시 실행하고, 그 안에서 SDK가 또 3번 시도합니다.

| 노드의 `retry_on` | SDK가 올린 예외를 받은 노드 | 모델 서버 요청 |
|---|---|---|
| 정하지 않음(기본 판정) | 다시 시도함. OpenAI 예외는 기본 판정의 "재시도 안 함" 목록에 없습니다 | SDK 3번 x 노드 시도 수 (겹침) |
| `ConnectionError` | 다시 시도하지 않음. OpenAI 예외는 이름이 비슷해도 파이썬 내장 `ConnectionError`의 하위 클래스가 아닙니다 | SDK 3번만 |

아래 셀은 이 교안의 `llm`이 SDK 재시도를 몇 번 하는지와, OpenAI 연결 오류가 `ConnectionError`에 속하는지를 확인합니다.

In [ ]:
import openai

# 1. SDK 재시도 횟수: llm에 max_retries를 정하지 않으면 OpenAI 클라이언트 기본값 2(시도 3번)를 씁니다.
print("ChatOpenAI max_retries:", llm.max_retries, "| OpenAI 클라이언트 max_retries:", llm.root_client.max_retries)

# 2. retry_on=ConnectionError가 OpenAI 연결 오류를 잡는지: 하위 클래스가 아니면 노드는 다시 시도하지 않습니다.
print("openai.APIConnectionError가 ConnectionError인가:", issubclass(openai.APIConnectionError, ConnectionError))

**확인할 점:** `ChatOpenAI max_retries:`는 `None`이지만 `OpenAI 클라이언트 max_retries:`는 2입니다. 그래서 모델 호출 한 번이 최대 3번 요청됩니다. `openai.APIConnectionError가 ConnectionError인가:`가 `False`이므로 `retry_on=ConnectionError`인 노드는 모델 연결 오류를 다시 시도하지 않고 SDK 재시도에만 맡깁니다.

5-2의 재시도 예산에도 SDK 시도를 넣습니다. 모델을 부르는 노드의 시도 한 번 안에는 SDK 요청 최대 3번과 그 사이 대기가 들어 있으므로 시도 한 번의 시간을 그만큼 길게 잡고, 요청 수(비용과 요청 한도)는 노드 시도 수 x SDK 시도 수로 셉니다.

**실무 원칙**

- **재시도는 한 계층에 맡깁니다.** 노드 `RetryPolicy`로 모델 오류까지 재시도하려면 `ChatOpenAI(model=..., max_retries=0)`으로 SDK 재시도를 끕니다. SDK에 맡기면 노드의 `retry_on`에서 모델 오류를 뺍니다.
- **또는 오류 종류별로 계층을 나눕니다.** 연결 오류와 429 같은 일시 오류는 SDK가 재시도하고, 영구 오류는 노드에서 재시도하지 않고 기록합니다.
- 여러 계층의 재시도가 곱해지면 장애가 난 서버에 부하를 더 얹어 회복을 늦춥니다. AWS Well-Architected 프레임워크(REL05-BP03 Control and limit retry calls)도 이 증폭을 경고하고 한 계층에서만 재시도하라고 권합니다.

## 6. 핵심 정리

- **데드락과 같은 증상(핑퐁, 에러 없는 중단):** 동시에 만족할 수 없는 규칙은 두 노드의 핑퐁을 만듭니다. `recursion_limit`은 직접 정하는 마지막 상한이고, 그 전에 라운드 가드와 `RemainingSteps`로 정상 종료합니다. 대기 엣지는 목록의 갈래 하나가 실행되지 않으면 합치는 노드를 실행하지 않고 예외 없이 끝납니다. 갈래마다 보통 엣지로 잇고 합치는 노드를 `defer=True`로 등록하면, 그 실행의 다른 노드가 모두 끝난 뒤 한 번 실행됩니다.
- **재시도:** `RetryPolicy(max_attempts=3)`은 첫 시도 포함 3번이고, `retry_on`에 일시 오류(`ConnectionError`, `NodeTimeoutError`)만 적습니다(예: 가격 API 2번 실패 뒤 3번째 성공). 기본 판정은 `KeyError`·`ValueError` 같은 영구 오류를 재시도하지 않습니다.
- **타임아웃과 재시도 예산:** 외부 응답을 끝없이 기다리는 노드는 `timeout`으로 끊습니다. `timeout`은 async 노드에만 걸리고, sync 노드에 걸면 `compile()`에서 `ValueError`가 납니다. 최악 대기 = 시도 수 x 타임아웃 + 대기 합 + jitter 최대값(재시도마다 1초)으로 예산을 정합니다.
- **겹친 재시도:** `ChatOpenAI`는 `max_retries`를 정하지 않아도 OpenAI SDK가 시도 3번(재시도 2번)까지 보내고, Langfuse에는 generation 하나로 보입니다. 노드 `RetryPolicy`가 모델 오류도 재시도하면 시도 수가 곱해지므로(SDK 3번 x 노드 2번 = 6번) 재시도는 한 계층에 맡기거나 오류 종류별로 계층을 나눕니다.